**Extraction of subgraph**

In [1]:
# On importe les bibliothèques dont on aura besoin tout au long du notebook, puis
# on définit quelques constantes globales : l'URL de l'API Wikidata, les cinq
# propriétés qu'on veut extraire (réalisateur, acteur, genre, date de publication, pays d'origine) avec leurs codes
# Wikidata, et une couleur par type de relation pour la visualisation finale.

import requests
import csv
import json
import time
import pandas as pd
import networkx as nx
from pyvis.network import Network

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "WikidataSubgraph/1.0 (student project)"}

RELEVANT_PROPERTIES = {
    "P57":  "réalisateur",
    "P161": "acteur",
    "P136": "genre",
    "P577": "date de publication",
    "P495": "pays d'origine",
}

PROP_COLORS = {
    "réalisateur":         "#E85D24",
    "acteur":              "#3B8BD4",
    "genre":               "#1D9E75",
    "date de publication": "#BA7517",
    "pays d'origine":      "#7F77DD",
}

In [2]:
# On définit les trois fonctions qui vont chercher les données sur Wikidata.
# La première construit la requête pour une propriété donnée (ex. acteur), la
# deuxième l'envoie à l'API par pages de 200 résultats et récupère les triplets
# (film, propriété, valeur), la troisième nettoie le tout en enlevant les doublons
# et les entrées sans label lisible.

def build_query(prop_id, limit=200, offset=0):
    return f"""
    SELECT ?film ?filmLabel ?value ?valueLabel WHERE {{
      ?film wdt:P31 wd:Q11424 ;
            wdt:P577 ?date ;
            wdt:{prop_id} ?value .
      FILTER(YEAR(?date) >= 2010)
      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "fr,en" .
      }}
    }}
    LIMIT {limit}
    OFFSET {offset}
    """

def fetch_triplets(page_size=200, n_pages=3):
    all_triplets = []
    for prop_id, prop_label in RELEVANT_PROPERTIES.items():
        print(f"  Propriété : {prop_label} ({prop_id})")
        for page in range(n_pages):
            query = build_query(prop_id, limit=page_size, offset=page * page_size)
            try:
                resp = requests.get(
                    WIKIDATA_ENDPOINT,
                    params={"query": query, "format": "json"},
                    headers=HEADERS,
                    timeout=30
                )
                resp.raise_for_status()
                results = resp.json()["results"]["bindings"]
                if not results:
                    break
                for r in results:
                    all_triplets.append({
                        "sujet_id":    r["film"]["value"].split("/")[-1],
                        "sujet_label": r.get("filmLabel", {}).get("value", ""),
                        "prop_id":     prop_id,
                        "prop_label":  prop_label,
                        "objet_id":    r["value"]["value"].split("/")[-1],
                        "objet_label": r.get("valueLabel", {}).get("value", ""),
                    })
                print(f"    page {page+1} : {len(results)} triplets")
                time.sleep(2)
            except requests.exceptions.Timeout:
                print(f"    Timeout sur {prop_label} page {page+1}, on passe.")
                time.sleep(5)
                break
            except Exception as e:
                print(f"    Erreur : {e}")
                time.sleep(5)
                break
    return all_triplets

def filter_triplets(triplets):
    filtered, seen = [], set()
    for t in triplets:
        key = (t["sujet_id"], t["prop_id"], t["objet_id"])
        if key in seen:
            continue
        seen.add(key)
        if not t["sujet_label"] or not t["objet_label"]:
            continue
        if t["objet_label"].startswith("Q") and t["objet_label"][1:].isdigit():
            continue
        filtered.append(t)
    return filtered

In [3]:
# On lance la collecte des données en appelant les fonctions définies juste avant,
# et on affiche le nombre de triplets récupérés avant et après nettoyage.

print("Extraction Wikidata – Cinéma (2010+)")
raw = fetch_triplets(page_size=200, n_pages=3)
print(f"\nTriplets bruts  : {len(raw)}")
clean = filter_triplets(raw)
print(f"Après filtrage  : {len(clean)}")

Extraction Wikidata – Cinéma (2010+)
  Propriété : réalisateur (P57)
    page 1 : 200 triplets
    page 2 : 200 triplets
    Timeout sur réalisateur page 3, on passe.
  Propriété : acteur (P161)
    page 1 : 200 triplets
    page 2 : 200 triplets
    page 3 : 200 triplets
  Propriété : genre (P136)
    page 1 : 200 triplets
    page 2 : 200 triplets
    Timeout sur genre page 3, on passe.
  Propriété : date de publication (P577)
    Timeout sur date de publication page 1, on passe.
  Propriété : pays d'origine (P495)
    page 1 : 200 triplets
    page 2 : 200 triplets
    page 3 : 200 triplets

Triplets bruts  : 2000
Après filtrage  : 759


In [4]:
# On convertit les triplets en tableau pandas et on sauvegarde le tout dans un
# fichier CSV, puis on affiche les 5 premières lignes pour vérifier que c'est bon.

df = pd.DataFrame(clean)
df.to_csv("subgraph.csv", index=False)
print(f"✓ {len(df)} triplets sauvegardés → subgraph.csv")
df.head()

✓ 759 triplets sauvegardés → subgraph.csv


,sujet_id,sujet_label,prop_id,prop_label,objet_id,objet_label
0,Q31540,Assassination Games,P57,réalisateur,Q2326442,Ernie Barbarash
1,Q24196,Inside,P57,réalisateur,Q3616495,Andrés Baiz
2,Q24515,Shaolin,P57,réalisateur,Q702326,Benny Chan
3,Q24871,Avatar,P57,réalisateur,Q42574,James Cameron
4,Q25157,Aliens vs Avatars,P57,réalisateur,Q23771513,Lewis Schoenbrun


In [5]:
# On recharge le CSV, on règle le problème des films avec plusieurs dates de
# sortie en ne gardant que la plus ancienne, puis on affiche quelques stats
# pour vérifier que les données sont cohérentes avant de construire le graphe.

df = pd.read_csv("subgraph.csv")
print(f"Triplets bruts chargés : {len(df)}\n")

# Nettoyer les doublons de dates (garder la plus ancienne = sortie originale)
dates = df[df["prop_label"] == "date de publication"].copy()
dates["objet_label"] = pd.to_datetime(dates["objet_label"], errors="coerce")
dates = dates.sort_values("objet_label").drop_duplicates(subset=["sujet_id"], keep="first")
dates["objet_label"] = dates["objet_label"].dt.strftime("%Y-%m-%d")

# Réassembler avec les autres propriétés
autres = df[df["prop_label"] != "date de publication"]
df = pd.concat([autres, dates], ignore_index=True)
print(f"Triplets après nettoyage des dates : {len(df)}\n")

print("=== Distribution par propriété ===")
print(df["prop_label"].value_counts())
print("\n=== Top 10 films (nb de triplets) ===")
print(df["sujet_label"].value_counts().head(10))
print("\n=== Top 10 objets les plus cités ===")
print(df["objet_label"].value_counts().head(10))

Triplets bruts chargés : 759

Triplets après nettoyage des dates : 759

=== Distribution par propriété ===
acteur            245
pays d'origine    196
genre             167
réalisateur       151
Name: prop_label, dtype: int64

=== Top 10 films (nb de triplets) ===
Argo                 41
Love and Secrets     32
La Chasse            30
Sept Psychopathes    28
Kick-Ass             27
Intouchables         25
Les Lyonnais         22
Lincoln              18
The Company Men      15
Apocalypse Now       14
Name: sujet_label, dtype: int64

=== Top 10 objets les plus cités ===
États-Unis                             82
drame                                  29
Royaume-Uni                            22
film et téléfilm d'action              18
Inde                                   16
France                                 11
film ou téléfilm d'aventure            10
thriller                                9
film ou téléfilm de science-fiction     9
comédie                                 8
Name:

In [6]:
# On construit le graphe à partir du DataFrame en ajoutant un nœud par film et
# par valeur (acteur, genre...), puis une arête entre les deux. Comme le graphe
# complet serait illisible, on le réduit aux 100 films les mieux connectés en
# gardant quand même tous leurs voisins directs.

G = nx.DiGraph()

for _, row in df.iterrows():
    subj = row["sujet_label"]
    obj  = row["objet_label"]
    prop = row["prop_label"]
    col  = PROP_COLORS.get(prop, "#888780")

    if not G.has_node(subj):
        G.add_node(subj, color="#F5C4B3", size=12, title=f"Film : {subj}", group="film")
    if not G.has_node(obj):
        G.add_node(obj, color=col, size=8, title=f"{prop} : {obj}", group=prop)
    G.add_edge(subj, obj, label=prop, color=col, width=1.5)

print(f"Nœuds : {G.number_of_nodes()}  |  Arêtes : {G.number_of_edges()}")

# Réduire aux 100 films les mieux connectés pour la lisibilité
top_films = sorted(
    [n for n, d in G.nodes(data=True) if d.get("group") == "film"],
    key=lambda n: G.degree(n),
    reverse=True
)[:100]

subgraph_nodes = set(top_films)
for film in top_films:
    subgraph_nodes.update(G.successors(film))

G_small = G.subgraph(subgraph_nodes).copy()
print(f"Sous-graphe réduit : {G_small.number_of_nodes()} nœuds, {G_small.number_of_edges()} arêtes")

Nœuds : 667  |  Arêtes : 757
Sous-graphe réduit : 543 nœuds, 659 arêtes


In [7]:
# On génère la visualisation interactive du graphe avec pyvis, on règle quelques
# paramètres physiques pour que les nœuds se répartissent lisiblement, puis on
# sauvegarde le tout dans un fichier HTML et on l'affiche directement dans le notebook.

from IPython.display import IFrame

net = Network(
    height="750px",
    width="100%",
    directed=True,
    bgcolor="#1a1a2e",
    font_color="white"
)
net.from_nx(G_small)

net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -60,
      "centralGravity": 0.005,
      "springLength": 100,
      "springConstant": 0.08,
      "damping": 0.4
    },
    "solver": "forceAtlas2Based",
    "stabilization": { "iterations": 150 }
  },
  "edges": {
    "smooth": { "type": "continuous" },
    "arrows": { "to": { "enabled": true, "scaleFactor": 0.5 } }
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "navigationButtons": true
  }
}
""")

net.save_graph("graph_cinema.html")
IFrame("graph_cinema.html", width="100%", height="750px")

**Generate candidates**

In [14]:
# On construit un index BM25 sur les entités du subgraphe pour retrouver
# les candidats les plus proches d'une mention par similarité lexicale.
# BM25 sert de baseline rapide avant l'approche par embeddings.

from rank_bm25 import BM25Okapi

def build_bm25_index(G):
    entities = list(G.nodes())
    tokenized = [e.lower().split() for e in entities]
    bm25 = BM25Okapi(tokenized)
    return bm25, entities

def get_candidates_bm25(mention, bm25, entities, N=5):
    tokens = mention.lower().split()  # bien mettre en lowercase ici aussi
    scores = bm25.get_scores(tokens)
    top_n = sorted(zip(entities, scores), key=lambda x: x[1], reverse=True)[:N]
    return top_n

# Utilisation
bm25, entities = build_bm25_index(G_small)
print(get_candidates_bm25("Avatar", bm25, entities, N=5))

[('Avatar', 7.920035904023279), ('Inside', 0.0), ('Andrés Baiz', 0.0), ('Shaolin', 0.0), ('Benny Chan', 0.0)]


In [9]:
print(entities[:20])

['Inside', 'Andrés Baiz', 'Shaolin', 'Benny Chan', 'Avatar', 'James Cameron', 'Aliens vs Avatars', 'Lewis Schoenbrun', 'Inception', 'Christopher Nolan', 'Ammavin Kaipesi', 'Thangar Bachan', 'Aarohanam', 'Lakshmy Ramakrishnan', 'Trois Mondes', 'Catherine Corsini', 'Hatchi', 'Lasse Hallström', 'The Mortal Instruments : La Cité des ténèbres', 'Harald Zwart']


In [10]:
print(get_candidates_bm25("Christopher Nolan", bm25, entities, N=5))

[('Christopher Nolan', 10.62566942929525), ('Matt Nolan', 5.736285890369083), ('Christopher Mintz-Plasse', 4.889383538926166), ('Christopher Walken', 4.889383538926166), ('Christopher Denham', 4.889383538926166)]


In [11]:
print(get_candidates_bm25("Cristopher Nolan", bm25, entities, N=5))
print(get_candidates_bm25("Nolan", bm25, entities, N=5))

[('Christopher Nolan', 5.736285890369083), ('Matt Nolan', 5.736285890369083), ('Inside', 0.0), ('Andrés Baiz', 0.0), ('Shaolin', 0.0)]
[('Christopher Nolan', 5.736285890369083), ('Matt Nolan', 5.736285890369083), ('Inside', 0.0), ('Andrés Baiz', 0.0), ('Shaolin', 0.0)]


On voit que Cristopher Nolan est "aussi proche" que Matt Nolan

In [12]:
# On encode toutes les entités du subgraphe en vecteurs avec un modèle
# d'embeddings, puis on construit un index FAISS pour retrouver rapidement
# les N entités les plus proches d'une mention par similarité sémantique.

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

def build_faiss_index(G):
    entities = list(G.nodes())
    
    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    print("Encodage des entités...")
    embeddings = model.encode(
        entities,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=64        # réduit la consommation mémoire
    )
    embeddings = np.array(embeddings, dtype="float32")
    
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    
    return index, entities, model

def get_candidates_faiss(mention, index, entities, model, N=5):
    emb = model.encode([mention], normalize_embeddings=True)
    emb = np.array(emb, dtype="float32")
    scores, indices = index.search(emb, N)
    return [(entities[i], scores[0][j]) for j, i in enumerate(indices[0])]


index, entities, model = build_faiss_index(G_small)
print(get_candidates_faiss("Leonardo DiCaprio", index, entities, model, N=5))

c:\Users\carle\anaconda3\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
c:\Users\carle\anaconda3\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Encodage des entités...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

[('Nicolas Cage', 0.53809017), ('Robert Capron', 0.5254462), ('Ryan Gosling', 0.5120379), ('Alfonso Cuarón', 0.51001674), ('James Cameron', 0.50859845)]


In [15]:
# Cas 1 : correspondance exacte
print(get_candidates_faiss("Christopher Nolan", index, entities, model, N=5))

# Cas 2 : faute d'orthographe (là où BM25 échoue)
print(get_candidates_faiss("Cristopher Nolan", index, entities, model, N=5))

# Cas 3 : nom partiel
print(get_candidates_faiss("Nolan", index, entities, model, N=5))

# Cas 4 : description sémantique sans le nom
print(get_candidates_faiss("réalisateur britannique", index, entities, model, N=20))

[('Christopher Nolan', 1.0), ('Matt Nolan', 0.806399), ('Christopher Stanley', 0.5913135), ('Matthew Vaughn', 0.55414367), ('Paul Freeman', 0.5507598)]
[('Christopher Nolan', 0.7814181), ('Matt Nolan', 0.77789307), ('Bryan Cranston', 0.5544814), ('Paul Freeman', 0.5251323), ('Frank Langella', 0.5148584)]
[('Matt Nolan', 0.9062495), ('Christopher Nolan', 0.83851814), ('Paul Freeman', 0.5709969), ('Thomas Bo Larsen', 0.50164247), ('Paul Sanchez', 0.49563482)]
[('Christophe Kourotchkine', 0.52663416), ('Le Dernier Rempart', 0.49986923), ('film ou téléfilm biographique', 0.46539724), ('Jack le chasseur de géants', 0.440891), ('Gilles Bénichou', 0.43821558), ('Marie-Laure Descoureaux', 0.42933404), ("Le Livre d'Eli", 0.42873842), ('drame psychologique', 0.42801893), ("Le Joyeux Noël d'Harold et Kumar", 0.42678723), ('Jean-François Cayrey', 0.42529988), ("L'Étrange Pouvoir de Norman", 0.4213863), ("Harry Potter et le Prisonnier d'Azkaban", 0.42094535), ("Le Dernier Maître de l'air", 0.415994